In [66]:
# ============================================================
# EXPERIMENT 2
# NAME-DOMINANT MULTI-PASS BLOCKING
#
# SELF-CONTAINED SETUP
# ============================================================

import pandas as pd
import numpy as np

import re
import unicodedata

from pathlib import Path
from collections import defaultdict, Counter

from anyascii import anyascii

from rapidfuzz import process, fuzz
from rapidfuzz.distance import Levenshtein

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# CONFIGURATION
# ============================================================

BASE_DIR = Path("sampled_data")

S1_FILE = BASE_DIR / "sample_source1.tsv"
S2_FILE = BASE_DIR / "sample_source2.tsv"
S3_FILE = BASE_DIR / "sample_source3.tsv"
GT_FILE = BASE_DIR / "sample_ground_truth.tsv"

OUTPUT_DIR = Path("blocking_results")
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Blocking parameters
# ------------------------------------------------------------

NAME_CHAR_TOP_K = 50
NAME_FUZZY_TOP_K = 50
NAME_TOKEN_TOP_K = 50

ADDRESS_TOP_K = 30
ADDRESS_OVERLAP_THRESHOLD = 2

NAME_CHAR_NGRAM_RANGE = (2, 5)
NAME_CHAR_MIN_DF = 1

NAME_FUZZY_THRESHOLD = 0.55


# ------------------------------------------------------------
# Weak address tokens
# ------------------------------------------------------------

WEAK_ADDRESS_TOKENS = {
    "road", "rd",
    "street", "st",
    "avenue", "ave",
    "lane", "ln",
    "drive", "dr",
    "highway", "hwy",
    "boulevard", "blvd",
    "way",
    "place", "pl",
    "parkway", "pkwy",
    "unit",
    "suite", "ste",
    "apt", "apartment",
    "floor", "fl",
    "building", "bldg",
    "block",
    "district",
    "city",
    "state",
    "county",
    "india",
    "usa",
    "us",
}


MIN_TOKEN_LENGTH = 3


# ------------------------------------------------------------
# Name legal suffixes
# ------------------------------------------------------------

LEGAL_SUFFIXES = {
    "limited",
    "ltd",
    "llc",
    "inc",
    "incorporated",
    "corp",
    "corporation",
    "company",
    "co",
    "private",
    "pvt",
    "plc",
    "llp",
    "lp",
}


print("=" * 80)
print("EXPERIMENT 2 — NAME-DOMINANT MULTI-PASS BLOCKING")
print("=" * 80)

print(f"S1 file: {S1_FILE}")
print(f"S2 file: {S2_FILE}")
print(f"S3 file: {S3_FILE}")
print(f"GT file: {GT_FILE}")

print()
print(f"Name char TF-IDF TOP-K: {NAME_CHAR_TOP_K}")
print(f"Name fuzzy TOP-K:       {NAME_FUZZY_TOP_K}")
print(f"Name token TOP-K:       {NAME_TOKEN_TOP_K}")
print(f"Address TOP-K:          {ADDRESS_TOP_K}")
print(f"Address overlap:        {ADDRESS_OVERLAP_THRESHOLD}")
print(f"Fuzzy threshold:        {NAME_FUZZY_THRESHOLD}")

EXPERIMENT 2 — NAME-DOMINANT MULTI-PASS BLOCKING
S1 file: sampled_data\sample_source1.tsv
S2 file: sampled_data\sample_source2.tsv
S3 file: sampled_data\sample_source3.tsv
GT file: sampled_data\sample_ground_truth.tsv

Name char TF-IDF TOP-K: 50
Name fuzzy TOP-K:       50
Name token TOP-K:       50
Address TOP-K:          30
Address overlap:        2
Fuzzy threshold:        0.55


In [67]:
# ============================================================
# EXPERIMENT 2
# NAME-DOMINANT MULTI-PASS BLOCKING
#
# SELF-CONTAINED SETUP
# ============================================================

import pandas as pd
import numpy as np

import re
import unicodedata

from pathlib import Path
from collections import defaultdict, Counter

from anyascii import anyascii

from rapidfuzz import process, fuzz
from rapidfuzz.distance import Levenshtein

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# CONFIGURATION
# ============================================================

BASE_DIR = Path("sampled_data")

S1_FILE = BASE_DIR / "sample_source1.tsv"
S2_FILE = BASE_DIR / "sample_source2.tsv"
S3_FILE = BASE_DIR / "sample_source3.tsv"
GT_FILE = BASE_DIR / "sample_ground_truth.tsv"

OUTPUT_DIR = Path("blocking_results")
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Blocking parameters
# ------------------------------------------------------------

NAME_CHAR_TOP_K = 50
NAME_FUZZY_TOP_K = 50
NAME_TOKEN_TOP_K = 50

ADDRESS_TOP_K = 30
ADDRESS_OVERLAP_THRESHOLD = 2

NAME_CHAR_NGRAM_RANGE = (2, 5)
NAME_CHAR_MIN_DF = 1

NAME_FUZZY_THRESHOLD = 0.55


# ------------------------------------------------------------
# Weak address tokens
# ------------------------------------------------------------

WEAK_ADDRESS_TOKENS = {
    "road", "rd",
    "street", "st",
    "avenue", "ave",
    "lane", "ln",
    "drive", "dr",
    "highway", "hwy",
    "boulevard", "blvd",
    "way",
    "place", "pl",
    "parkway", "pkwy",
    "unit",
    "suite", "ste",
    "apt", "apartment",
    "floor", "fl",
    "building", "bldg",
    "block",
    "district",
    "city",
    "state",
    "county",
    "india",
    "usa",
    "us",
}


MIN_TOKEN_LENGTH = 3


# ------------------------------------------------------------
# Name legal suffixes
# ------------------------------------------------------------

LEGAL_SUFFIXES = {
    "limited",
    "ltd",
    "llc",
    "inc",
    "incorporated",
    "corp",
    "corporation",
    "company",
    "co",
    "private",
    "pvt",
    "plc",
    "llp",
    "lp",
}


print("=" * 80)
print("EXPERIMENT 2 — NAME-DOMINANT MULTI-PASS BLOCKING")
print("=" * 80)

print(f"S1 file: {S1_FILE}")
print(f"S2 file: {S2_FILE}")
print(f"S3 file: {S3_FILE}")
print(f"GT file: {GT_FILE}")

print()
print(f"Name char TF-IDF TOP-K: {NAME_CHAR_TOP_K}")
print(f"Name fuzzy TOP-K:       {NAME_FUZZY_TOP_K}")
print(f"Name token TOP-K:       {NAME_TOKEN_TOP_K}")
print(f"Address TOP-K:          {ADDRESS_TOP_K}")
print(f"Address overlap:        {ADDRESS_OVERLAP_THRESHOLD}")
print(f"Fuzzy threshold:        {NAME_FUZZY_THRESHOLD}")

EXPERIMENT 2 — NAME-DOMINANT MULTI-PASS BLOCKING
S1 file: sampled_data\sample_source1.tsv
S2 file: sampled_data\sample_source2.tsv
S3 file: sampled_data\sample_source3.tsv
GT file: sampled_data\sample_ground_truth.tsv

Name char TF-IDF TOP-K: 50
Name fuzzy TOP-K:       50
Name token TOP-K:       50
Address TOP-K:          30
Address overlap:        2
Fuzzy threshold:        0.55


In [68]:
# ============================================================
# LOAD DATA
# ============================================================

print("=" * 80)
print("LOADING DATA")
print("=" * 80)


s1 = pd.read_csv(
    S1_FILE,
    sep="\t",
    dtype=str
).fillna("")


s2 = pd.read_csv(
    S2_FILE,
    sep="\t",
    dtype=str
).fillna("")


s3 = pd.read_csv(
    S3_FILE,
    sep="\t",
    dtype=str
).fillna("")


gt = pd.read_csv(
    GT_FILE,
    sep="\t",
    dtype=str
).fillna("")


print(f"S1 shape: {s1.shape}")
print(f"S2 shape: {s2.shape}")
print(f"S3 shape: {s3.shape}")
print(f"GT shape: {gt.shape}")


print("\nColumns:")
print("S1:", list(s1.columns))
print("S2:", list(s2.columns))
print("S3:", list(s3.columns))
print("GT:", list(gt.columns))


required_source_columns = {
    "entity_id",
    "business_name",
    "business_address",
    "country"
}


for source_name, df in {
    "S1": s1,
    "S2": s2,
    "S3": s3
}.items():

    missing = (
        required_source_columns
        -
        set(df.columns)
    )

    if missing:
        raise ValueError(
            f"{source_name} is missing columns: {missing}"
        )


required_gt_columns = {
    "source1_entity_id",
    "matched_entity_ids"
}


missing_gt = (
    required_gt_columns
    -
    set(gt.columns)
)


if missing_gt:
    raise ValueError(
        f"Ground truth is missing columns: {missing_gt}"
    )


print("\nData validation passed.")

LOADING DATA
S1 shape: (1000, 4)
S2 shape: (9864, 4)
S3 shape: (10842, 4)
GT shape: (1000, 2)

Columns:
S1: ['entity_id', 'business_name', 'business_address', 'country']
S2: ['entity_id', 'business_name', 'business_address', 'country']
S3: ['entity_id', 'business_name', 'business_address', 'country']
GT: ['source1_entity_id', 'matched_entity_ids']

Data validation passed.


In [69]:
# ============================================================
# TEXT NORMALIZATION HELPERS
# ============================================================

def is_latin_char(ch):
    """
    Return True if a character belongs to the Latin script.
    """

    try:
        unicode_name = unicodedata.name(ch)
    except ValueError:
        return False

    return "LATIN" in unicode_name


def contains_non_latin(text):
    """
    Detect alphabetic characters outside Latin script.
    """

    if not isinstance(text, str):
        return False

    for ch in text:

        if (
            ch.isalpha()
            and not is_latin_char(ch)
        ):
            return True

    return False


def transliterate_text(text):
    """
    Convert text to a Latin-oriented representation.
    """

    if not isinstance(text, str):
        return ""

    if not text:
        return ""

    return anyascii(text)


def normalize_text(
    text,
    transliterate=False
):
    """
    General normalization.

    Steps:
    - Unicode normalization
    - lowercase
    - optional transliteration
    - punctuation -> spaces
    - whitespace normalization
    """

    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize(
        "NFKC",
        text
    )

    text = text.lower().strip()

    if transliterate:
        text = transliterate_text(text)

    text = re.sub(
        r"[^\w\s]",
        " ",
        text,
        flags=re.UNICODE
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def tokenize(text):
    """
    Normalize whitespace-separated tokens.
    """

    if not text:
        return []

    return text.split()


def transliterated_tokens(text):
    """
    Normalize and transliterate tokens.
    """

    normalized = normalize_text(
        text,
        transliterate=False
    )

    result = []

    for token in tokenize(normalized):

        if contains_non_latin(token):
            token = transliterate_text(token)

        token = normalize_text(
            token,
            transliterate=False
        )

        if token:
            result.append(token)

    return result


def get_informative_address_tokens(address):
    """
    Extract useful address tokens.

    Weak/common structural address words are removed.
    """

    tokens = transliterated_tokens(address)

    informative = set()

    for token in tokens:

        if len(token) < MIN_TOKEN_LENGTH:
            continue

        if token in WEAK_ADDRESS_TOKENS:
            continue

        if not re.search(
            r"[a-z0-9]",
            token
        ):
            continue

        informative.add(token)

    return informative


def get_name_tokens(name):
    """
    Full normalized/transliterated name tokens.
    """

    return transliterated_tokens(name)


def get_name_core_tokens(name):
    """
    Name tokens with common legal suffixes removed.
    """

    tokens = get_name_tokens(name)

    return [
        token
        for token in tokens
        if token not in LEGAL_SUFFIXES
    ]


def get_name_sorted_text(name):
    """
    Order-independent name representation.
    """

    tokens = get_name_tokens(name)

    return " ".join(
        sorted(tokens)
    )


def get_name_core_text(name):
    """
    Name representation with legal suffixes removed.
    """

    tokens = get_name_core_tokens(name)

    return " ".join(tokens)


def get_name_char_text(name):
    """
    Transliteration-aware normalized name text.
    """

    return normalize_text(
        name,
        transliterate=True
    )


def token_jaccard(
    left,
    right
):
    """
    Jaccard similarity between two token collections.
    """

    left = set(left)
    right = set(right)

    if not left and not right:
        return 1.0

    if not left or not right:
        return 0.0

    return len(
        left & right
    ) / len(
        left | right
    )


def build_address_tokens(df):

    df = df.copy()

    df["address_tokens"] = (
        df["business_address"]
        .apply(
            get_informative_address_tokens
        )
    )

    return df


def prepare_name_features(df):

    df = df.copy()

    df["name_tokens"] = (
        df["business_name"]
        .apply(get_name_tokens)
    )

    df["name_core_tokens"] = (
        df["business_name"]
        .apply(get_name_core_tokens)
    )

    df["name_sorted"] = (
        df["business_name"]
        .apply(get_name_sorted_text)
    )

    df["name_core_text"] = (
        df["business_name"]
        .apply(get_name_core_text)
    )

    df["name_char_text"] = (
        df["business_name"]
        .apply(get_name_char_text)
    )

    return df


def prepare_country(df):

    df = df.copy()

    df["country_norm"] = (
        df["country"]
        .apply(
            lambda x:
            normalize_text(
                x,
                transliterate=True
            )
        )
    )

    return df


print("All helper functions declared.")

All helper functions declared.


In [70]:
# ============================================================
# PREPARE NORMALIZED REPRESENTATIONS
# ============================================================

print("=" * 80)
print("PREPARING REPRESENTATIONS")
print("=" * 80)


s1 = prepare_country(s1)
s2 = prepare_country(s2)
s3 = prepare_country(s3)


s1 = prepare_name_features(s1)
s2 = prepare_name_features(s2)
s3 = prepare_name_features(s3)


s1 = build_address_tokens(s1)
s2 = build_address_tokens(s2)
s3 = build_address_tokens(s3)


print("S1 representations ready.")
print("S2 representations ready.")
print("S3 representations ready.")


print()
print("S1 columns added:")
print([
    "country_norm",
    "name_tokens",
    "name_core_tokens",
    "name_sorted",
    "name_core_text",
    "name_char_text",
    "address_tokens"
])

PREPARING REPRESENTATIONS
S1 representations ready.
S2 representations ready.
S3 representations ready.

S1 columns added:
['country_norm', 'name_tokens', 'name_core_tokens', 'name_sorted', 'name_core_text', 'name_char_text', 'address_tokens']


In [71]:
# ============================================================
# GROUND TRUTH PARSING
# ============================================================

def parse_ground_truth(gt_df):

    rows = []

    for _, row in gt_df.iterrows():

        s1_id = str(
            row["source1_entity_id"]
        ).strip()

        matched = str(
            row["matched_entity_ids"]
        ).strip()

        if not matched:
            continue

        for candidate_id in matched.split(","):

            candidate_id = (
                candidate_id.strip()
            )

            if candidate_id:

                rows.append({
                    "s1_entity_id": s1_id,
                    "true_entity_id": candidate_id
                })

    return pd.DataFrame(
        rows,
        columns=[
            "s1_entity_id",
            "true_entity_id"
        ]
    )


gt_pairs = parse_ground_truth(gt)


print("=" * 80)
print("GROUND TRUTH")
print("=" * 80)

print(
    f"GT S1 rows: {len(gt):,}"
)

print(
    f"Positive pairs: {len(gt_pairs):,}"
)

print(
    f"S1 entities with matches: "
    f"{gt_pairs['s1_entity_id'].nunique():,}"
)

singleton_count = (
    len(s1)
    -
    gt_pairs["s1_entity_id"].nunique()
)

print(
    f"Singleton S1 entities: "
    f"{singleton_count:,}"
)

print(
    f"Singleton percentage: "
    f"{singleton_count / len(s1):.2%}"
)

GROUND TRUTH
GT S1 rows: 1,000
Positive pairs: 3,451
S1 entities with matches: 934
Singleton S1 entities: 66
Singleton percentage: 6.60%


In [72]:
# ============================================================
# BUILD S2 + S3 CANDIDATE POPULATION
# ============================================================

source_records = pd.concat(
    [
        s2.assign(source="S2"),
        s3.assign(source="S3")
    ],
    ignore_index=True
).copy()


print("=" * 80)
print("S2 + S3 CANDIDATE POPULATION")
print("=" * 80)

print(
    f"S2 records: {len(s2):,}"
)

print(
    f"S3 records: {len(s3):,}"
)

print(
    f"Total source records: "
    f"{len(source_records):,}"
)


duplicate_entity_ids = (
    source_records["entity_id"]
    .duplicated()
    .sum()
)

print(
    f"Duplicate S2/S3 entity IDs: "
    f"{duplicate_entity_ids:,}"
)


if duplicate_entity_ids:
    raise ValueError(
        "Duplicate S2/S3 entity IDs found."
    )

S2 + S3 CANDIDATE POPULATION
S2 records: 9,864
S3 records: 10,842
Total source records: 20,706
Duplicate S2/S3 entity IDs: 0


In [73]:
# ============================================================
# LOOKUPS
# ============================================================

source_lookup = (
    source_records
    .set_index("entity_id")
    .to_dict("index")
)


s1_lookup = (
    s1
    .set_index("entity_id")
    .to_dict("index")
)


print(
    f"S1 lookup records: "
    f"{len(s1_lookup):,}"
)

print(
    f"S2/S3 lookup records: "
    f"{len(source_lookup):,}"
)

S1 lookup records: 1,000
S2/S3 lookup records: 20,706


In [74]:
# ============================================================
# ADDRESS INVERTED INDEX
# ============================================================

address_index = defaultdict(set)


for _, row in source_records.iterrows():

    entity_id = row["entity_id"]

    for token in row["address_tokens"]:

        address_index[token].add(
            entity_id
        )


print("=" * 80)
print("ADDRESS INDEX")
print("=" * 80)

print(
    f"Unique address tokens: "
    f"{len(address_index):,}"
)

ADDRESS INDEX
Unique address tokens: 25,584


In [75]:
# ============================================================
# NAME TOKEN INDEXES
# ============================================================

name_token_index = defaultdict(set)

name_core_token_index = defaultdict(set)


for _, row in source_records.iterrows():

    entity_id = row["entity_id"]

    for token in set(
        row["name_tokens"]
    ):

        if len(token) >= 2:

            name_token_index[
                token
            ].add(entity_id)


    for token in set(
        row["name_core_tokens"]
    ):

        if len(token) >= 2:

            name_core_token_index[
                token
            ].add(entity_id)


print("=" * 80)
print("NAME TOKEN INDEXES")
print("=" * 80)

print(
    f"Name tokens: "
    f"{len(name_token_index):,}"
)

print(
    f"Core-name tokens: "
    f"{len(name_core_token_index):,}"
)

NAME TOKEN INDEXES
Name tokens: 15,266
Core-name tokens: 15,253


In [76]:
# ============================================================
# COUNTRY-PARTITIONED FUZZY INDEX
# ============================================================

country_name_choices = defaultdict(dict)


for _, row in source_records.iterrows():

    country = row["country_norm"]

    entity_id = row["entity_id"]

    if not country:
        continue

    country_name_choices[
        country
    ][entity_id] = row[
        "name_char_text"
    ]


print("=" * 80)
print("FUZZY NAME INDEX")
print("=" * 80)

print(
    f"Country partitions: "
    f"{len(country_name_choices):,}"
)

FUZZY NAME INDEX
Country partitions: 2


In [77]:
# ============================================================
# NAME CHARACTER TF-IDF INDEX
# ============================================================

name_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=NAME_CHAR_NGRAM_RANGE,
    min_df=NAME_CHAR_MIN_DF,
    lowercase=False
)


source_name_matrix = (
    name_tfidf.fit_transform(
        source_records[
            "name_char_text"
        ]
    )
)


source_country_array = (
    source_records[
        "country_norm"
    ].to_numpy()
)


source_entity_array = (
    source_records[
        "entity_id"
    ].to_numpy()
)


print("=" * 80)
print("NAME CHAR-TFIDF INDEX")
print("=" * 80)

print(
    f"Matrix shape: "
    f"{source_name_matrix.shape}"
)

print(
    f"Vocabulary size: "
    f"{len(name_tfidf.vocabulary_):,}"
)
# ------------------------------------------------------------
# Build vectorizer
# ------------------------------------------------------------

name_char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=NAME_CHAR_NGRAM_RANGE,
    min_df=NAME_CHAR_MIN_DF,
    lowercase=False
)


# ------------------------------------------------------------
# Fit on all S2/S3 names
# ------------------------------------------------------------

source_name_char_texts = (
    source_records["name_char_text"]
    .fillna("")
    .astype(str)
    .tolist()
)


name_char_matrix = name_char_vectorizer.fit_transform(
    source_name_char_texts
)


# ------------------------------------------------------------
# Build country -> matrix row indices
# ------------------------------------------------------------

name_char_country_indices = {}

for idx, country in enumerate(
    source_records["country_norm"]
):

    name_char_country_indices.setdefault(
        country,
        []
    ).append(idx)


print("Name character TF-IDF index built.")
print(
    "Matrix shape:",
    name_char_matrix.shape
)
print(
    "Vocabulary size:",
    len(name_char_vectorizer.vocabulary_)
)
print(
    "Country partitions:",
    len(name_char_country_indices)
)

NAME CHAR-TFIDF INDEX
Matrix shape: (20706, 91906)
Vocabulary size: 91,906
Name character TF-IDF index built.
Matrix shape: (20706, 140780)
Vocabulary size: 140780
Country partitions: 2


In [78]:
# ============================================================
# BLOCKER A — ADDRESS
# ============================================================

def retrieve_address_candidates(
    s1_row
):

    s1_id = s1_row["entity_id"]

    country = s1_row[
        "country_norm"
    ]

    query_tokens = set(
        s1_row["address_tokens"]
    )

    if not query_tokens:
        return []

    counts = Counter()

    for token in query_tokens:

        candidate_ids = (
            address_index.get(
                token,
                set()
            )
        )

        for entity_id in candidate_ids:

            counts[entity_id] += 1


    ranked = []

    for entity_id, overlap in (
        counts.most_common()
    ):

        if len(ranked) >= ADDRESS_TOP_K:
            break

        candidate = source_lookup[
            entity_id
        ]

        if (
            candidate["country_norm"]
            != country
        ):
            continue

        if (
            overlap
            < ADDRESS_OVERLAP_THRESHOLD
        ):
            continue

        ranked.append({
            "s1_entity_id": s1_id,
            "candidate_entity_id": entity_id,
            "block_method": "address",
            "block_score": float(overlap)
        })

    return ranked

In [79]:
# ============================================================
# BLOCK 14 — NAME CHAR TF-IDF BLOCKER
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity


NAME_CHAR_TOP_K = 50
NAME_CHAR_MIN_SIMILARITY = 0.35


def retrieve_name_char_candidates(s1_row):
    """
    Retrieve S2/S3 candidates for one S1 row using
    character n-gram TF-IDF cosine similarity on business name.

    Returns:
        list of tuples:
        [(s1_id, source_id, similarity_score), ...]
    """

    s1_id = s1_row["entity_id"]
    country = s1_row["country_norm"]

    query_text = s1_row["name_char_text"]

    if not query_text:
        return []

    # --------------------------------------------------------
    # Get the S1 TF-IDF vector
    # --------------------------------------------------------
    query_vector = name_char_vectorizer.transform([query_text])

    # --------------------------------------------------------
    # Restrict retrieval to same-country records
    # --------------------------------------------------------
    candidate_indices = name_char_country_indices.get(country, [])

    if not candidate_indices:
        return []

    source_matrix = name_char_matrix[candidate_indices]

    # --------------------------------------------------------
    # Cosine similarity
    # --------------------------------------------------------
    similarities = cosine_similarity(
        query_vector,
        source_matrix
    ).ravel()

    # --------------------------------------------------------
    # Keep candidates above similarity threshold
    # --------------------------------------------------------
    valid_positions = [
        i
        for i, score in enumerate(similarities)
        if score >= NAME_CHAR_MIN_SIMILARITY
    ]

    if not valid_positions:
        return []

    # --------------------------------------------------------
    # Sort by similarity descending
    # --------------------------------------------------------
    valid_positions.sort(
        key=lambda i: similarities[i],
        reverse=True
    )

    # --------------------------------------------------------
    # Keep only top K
    # --------------------------------------------------------
    valid_positions = valid_positions[:NAME_CHAR_TOP_K]

    results = []

    for position in valid_positions:

        source_matrix_index = candidate_indices[position]

        source_id = source_records.iloc[
            source_matrix_index
        ]["entity_id"]

        similarity_score = float(
            similarities[position]
        )

        results.append(
            (
                s1_id,
                source_id,
                similarity_score
            )
        )

    return results


print("Name char TF-IDF blocker function defined.")
print(f"Top K: {NAME_CHAR_TOP_K}")
print(f"Minimum cosine similarity: {NAME_CHAR_MIN_SIMILARITY}")

Name char TF-IDF blocker function defined.
Top K: 50
Minimum cosine similarity: 0.35


In [80]:
# ============================================================
# BLOCKER B — NAME TOKEN
# ============================================================

def retrieve_name_token_candidates(
    s1_row
):

    s1_id = s1_row["entity_id"]

    country = s1_row[
        "country_norm"
    ]

    query_tokens = set(
        s1_row["name_tokens"]
    )

    query_core_tokens = set(
        s1_row["name_core_tokens"]
    )

    if not query_tokens:
        return []

    counts = Counter()


    # --------------------------------------------------------
    # Full-name tokens
    # --------------------------------------------------------

    for token in query_tokens:

        if len(token) < 2:
            continue

        for entity_id in (
            name_token_index.get(
                token,
                set()
            )
        ):

            counts[entity_id] += 1


    # --------------------------------------------------------
    # Core-name tokens
    # --------------------------------------------------------

    for token in query_core_tokens:

        if len(token) < 2:
            continue

        for entity_id in (
            name_core_token_index.get(
                token,
                set()
            )
        ):

            counts[entity_id] += 1


    ranked = []

    for entity_id, overlap in (
        counts.most_common()
    ):

        if len(ranked) >= NAME_TOKEN_TOP_K:
            break

        candidate = source_lookup[
            entity_id
        ]

        if (
            candidate["country_norm"]
            != country
        ):
            continue

        ranked.append({
            "s1_entity_id": s1_id,
            "candidate_entity_id": entity_id,
            "block_method": "name_token",
            "block_score": float(overlap)
        })

    return ranked

In [81]:
# ============================================================
# BLOCKER D — NAME FUZZY
# ============================================================

def retrieve_name_fuzzy_candidates(
    s1_row
):

    s1_id = s1_row["entity_id"]

    country = s1_row[
        "country_norm"
    ]

    query = s1_row[
        "name_char_text"
    ]

    if not country:
        return []

    if not query:
        return []


    choices = (
        country_name_choices.get(
            country,
            {}
        )
    )


    if not choices:
        return []


    matches = process.extract(
        query,
        choices,
        scorer=fuzz.ratio,
        limit=NAME_FUZZY_TOP_K
    )


    results = []


    for _, score, entity_id in matches:

        normalized_score = (
            float(score) / 100.0
        )

        if (
            normalized_score
            < NAME_FUZZY_THRESHOLD
        ):
            continue

        results.append({
            "s1_entity_id": s1_id,
            "candidate_entity_id": entity_id,
            "block_method": "name_fuzzy",
            "block_score": normalized_score
        })


    return results

In [82]:
# ============================================================
# RUN ALL BLOCKERS
# ============================================================

print("=" * 80)
print("RUNNING ALL BLOCKERS")
print("=" * 80)


address_rows = []
name_token_rows = []
name_char_rows = []
name_fuzzy_rows = []


total_s1 = len(s1)


for i, (_, row) in enumerate(
    s1.iterrows(),
    start=1
):

    if (
        i == 1
        or i % 100 == 0
        or i == total_s1
    ):

        print(
            f"Processing "
            f"S1 {i:,}/{total_s1:,}"
        )


    address_rows.extend(
        retrieve_address_candidates(
            row
        )
    )


    name_token_rows.extend(
        retrieve_name_token_candidates(
            row
        )
    )


    name_char_rows.extend(
        retrieve_name_char_candidates(
            row
        )
    )


    name_fuzzy_rows.extend(
        retrieve_name_fuzzy_candidates(
            row
        )
    )


address_candidates = pd.DataFrame(
    address_rows
)

name_token_candidates = pd.DataFrame(
    name_token_rows
)

name_char_candidates = pd.DataFrame(
    name_char_rows,
    columns=[
        "s1_entity_id",
        "candidate_entity_id",
        "block_score"
    ]
)

name_fuzzy_candidates = pd.DataFrame(
    name_fuzzy_rows
)


print("\nBlocking complete.")

print(
    f"Address rows: "
    f"{len(address_candidates):,}"
)

print(
    f"Name token rows: "
    f"{len(name_token_candidates):,}"
)

print(
    f"Name char TF-IDF rows: "
    f"{len(name_char_candidates):,}"
)

print(
    f"Name fuzzy rows: "
    f"{len(name_fuzzy_candidates):,}"
)

RUNNING ALL BLOCKERS
Processing S1 1/1,000
Processing S1 100/1,000
Processing S1 200/1,000
Processing S1 300/1,000
Processing S1 400/1,000
Processing S1 500/1,000
Processing S1 600/1,000
Processing S1 700/1,000
Processing S1 800/1,000
Processing S1 900/1,000
Processing S1 1,000/1,000

Blocking complete.
Address rows: 14,045
Name token rows: 46,730
Name char TF-IDF rows: 19,128
Name fuzzy rows: 39,899


In [83]:
# ============================================================
# BLOCK 17 — CANDIDATE SET HELPERS
# Handles BOTH DataFrames and tuple/list blocker outputs
# ============================================================

PAIR_COLUMNS = [
    "s1_entity_id",
    "candidate_entity_id"
]


def unique_pairs(candidate_data):
    """
    Normalize blocker output into:

        s1_entity_id
        candidate_entity_id

    Supported input formats:

    1. DataFrame:
       s1_entity_id, candidate_entity_id

    2. DataFrame:
       s1_id, source_id

    3. DataFrame:
       entity_id, candidate_id

    4. DataFrame:
       positional columns [0, 1, 2] from tuple output

    5. List of tuples:
       (s1_id, source_id)

    6. List of tuples:
       (s1_id, source_id, score)

    The third tuple element, if present, is ignored here because
    this function is only responsible for pair identity.
    """

    # --------------------------------------------------------
    # Empty / None
    # --------------------------------------------------------

    if candidate_data is None:
        return pd.DataFrame(columns=PAIR_COLUMNS)

    if isinstance(candidate_data, (list, tuple)):

        if len(candidate_data) == 0:
            return pd.DataFrame(columns=PAIR_COLUMNS)

        # ----------------------------------------------------
        # Convert tuple/list blocker output
        # ----------------------------------------------------

        rows = []

        for item in candidate_data:

            if not isinstance(item, (list, tuple)):
                raise TypeError(
                    "Expected each candidate to be a tuple/list, "
                    f"got: {type(item)}"
                )

            if len(item) < 2:
                raise ValueError(
                    "Candidate tuple must contain at least "
                    f"S1 ID and candidate ID. Got: {item}"
                )

            rows.append(
                {
                    "s1_entity_id": str(item[0]),
                    "candidate_entity_id": str(item[1])
                }
            )

        result = pd.DataFrame(rows)

    # --------------------------------------------------------
    # DataFrame input
    # --------------------------------------------------------

    elif isinstance(candidate_data, pd.DataFrame):

        if len(candidate_data) == 0:
            return pd.DataFrame(columns=PAIR_COLUMNS)

        df = candidate_data.copy()

        # Already standardized
        if set(PAIR_COLUMNS).issubset(df.columns):

            result = df[
                PAIR_COLUMNS
            ].copy()

        # s1_id / source_id
        elif {
            "s1_id",
            "source_id"
        }.issubset(df.columns):

            result = (
                df[
                    ["s1_id", "source_id"]
                ]
                .rename(
                    columns={
                        "s1_id": "s1_entity_id",
                        "source_id": "candidate_entity_id"
                    }
                )
            )

        # entity_id / candidate_id
        elif {
            "entity_id",
            "candidate_id"
        }.issubset(df.columns):

            result = (
                df[
                    ["entity_id", "candidate_id"]
                ]
                .rename(
                    columns={
                        "entity_id": "s1_entity_id",
                        "candidate_id": "candidate_entity_id"
                    }
                )
            )

        # Positional tuple DataFrame: (s1_id, source_id, score)
        elif len(df.columns) >= 2 and all(
            column in df.columns
            for column in [0, 1]
        ):

            result = df[
                [0, 1]
            ].rename(
                columns={
                    0: "s1_entity_id",
                    1: "candidate_entity_id"
                }
            )

        else:

            raise KeyError(
                "Could not identify S1/candidate columns.\n\n"
                f"Available columns: {list(df.columns)}"
            )

    else:

        raise TypeError(
            "Unsupported candidate output type: "
            f"{type(candidate_data)}"
        )

    # --------------------------------------------------------
    # Remove missing IDs
    # --------------------------------------------------------

    result = result.dropna(
        subset=PAIR_COLUMNS
    )

    # --------------------------------------------------------
    # Convert IDs to strings
    # --------------------------------------------------------

    result["s1_entity_id"] = (
        result["s1_entity_id"]
        .astype(str)
    )

    result["candidate_entity_id"] = (
        result["candidate_entity_id"]
        .astype(str)
    )

    # --------------------------------------------------------
    # Remove duplicate pairs
    # --------------------------------------------------------

    result = (
        result[
            PAIR_COLUMNS
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    return result


def union_candidate_sets(*candidate_dfs):
    """
    Union candidate outputs from multiple blockers.
    """

    cleaned = []

    for candidate_data in candidate_dfs:

        if candidate_data is None:
            continue

        pairs = unique_pairs(candidate_data)

        if len(pairs) > 0:
            cleaned.append(pairs)

    if not cleaned:

        return pd.DataFrame(
            columns=PAIR_COLUMNS
        )

    return (
        pd.concat(
            cleaned,
            ignore_index=True
        )
        .drop_duplicates(
            subset=PAIR_COLUMNS
        )
        .reset_index(drop=True)
    )


def make_pair_keys(candidate_data):
    """
    Return a set of (S1 ID, candidate ID) tuples.
    """

    pairs = unique_pairs(candidate_data)

    return set(
        zip(
            pairs["s1_entity_id"],
            pairs["candidate_entity_id"]
        )
    )


print("Candidate helper functions loaded successfully.")
print("PAIR_COLUMNS:", PAIR_COLUMNS)

Candidate helper functions loaded successfully.
PAIR_COLUMNS: ['s1_entity_id', 'candidate_entity_id']


In [84]:
# ============================================================
# BLOCKING EVALUATION
# ============================================================

def evaluate_blocking(
    candidate_df,
    method_name
):

    pairs = unique_pairs(
        candidate_df
    )


    total_true = len(gt_pairs)


    if total_true == 0:

        return {
            "method": method_name,
            "true_pairs": 0,
            "retrieved_true_pairs": 0,
            "missed_true_pairs": 0,
            "blocking_recall": 0.0,
            "candidate_pairs": len(pairs),
            "average_candidates_per_s1": 0.0,
            "median_candidates_per_s1": 0.0,
            "p95_candidates_per_s1": 0.0,
            "p99_candidates_per_s1": 0.0,
            "max_candidates_per_s1": 0
        }


    candidate_keys = make_pair_keys(
        pairs
    )


    gt_keys = set(
        zip(
            gt_pairs["s1_entity_id"].astype(str),
            gt_pairs["true_entity_id"].astype(str)
        )
    )


    retrieved = len(
        gt_keys & candidate_keys
    )


    counts = (
        pairs
        .groupby(
            "s1_entity_id"
        )
        .size()
        .reindex(
            s1["entity_id"],
            fill_value=0
        )
    )


    return {
        "method": method_name,

        "true_pairs": total_true,

        "retrieved_true_pairs":
            retrieved,

        "missed_true_pairs":
            total_true - retrieved,

        "blocking_recall":
            retrieved / total_true,

        "candidate_pairs":
            len(pairs),

        "average_candidates_per_s1":
            float(counts.mean()),

        "median_candidates_per_s1":
            float(counts.median()),

        "p95_candidates_per_s1":
            float(
                counts.quantile(0.95)
            ),

        "p99_candidates_per_s1":
            float(
                counts.quantile(0.99)
            ),

        "max_candidates_per_s1":
            int(counts.max())
    }


# ------------------------------------------------------------
# Individual blocker results
# ------------------------------------------------------------

individual_results = [

    evaluate_blocking(
        address_candidates,
        "address"
    ),

    evaluate_blocking(
        name_token_candidates,
        "name_token"
    ),

    evaluate_blocking(
        name_char_candidates,
        "name_char_tfidf"
    ),

    evaluate_blocking(
        name_fuzzy_candidates,
        "name_fuzzy"
    )
]


individual_comparison = (
    pd.DataFrame(
        individual_results
    )
)


individual_comparison[
    "recall_percent"
] = (
    individual_comparison[
        "blocking_recall"
    ] * 100
).round(3)


individual_comparison

,method,true_pairs,retrieved_true_pairs,missed_true_pairs,blocking_recall,candidate_pairs,average_candidates_per_s1,median_candidates_per_s1,p95_candidates_per_s1,p99_candidates_per_s1,max_candidates_per_s1,recall_percent
0,address,3451,3060,391,0.886700,14045,14.045,6.0,30.0,30.0,30,88.670
1,name_token,3451,2834,617,0.821211,46730,46.730,50.0,50.0,50.0,50,82.121
2,name_char_tfidf,3451,3060,391,0.886700,19128,19.128,11.0,50.0,50.0,50,88.670
3,name_fuzzy,3451,3097,354,0.897421,39899,39.899,50.0,50.0,50.0,50,89.742


In [85]:
# ============================================================
# UNION BLOCKING EXPERIMENTS
# ============================================================

name_union = union_candidate_sets(
    name_token_candidates,
    name_char_candidates,
    name_fuzzy_candidates
)


name_char_address = union_candidate_sets(
    name_char_candidates,
    address_candidates
)


name_fuzzy_address = union_candidate_sets(
    name_fuzzy_candidates,
    address_candidates
)


all_name_address = union_candidate_sets(
    name_union,
    address_candidates
)


union_results = [

    evaluate_blocking(
        name_char_address,
        "name_char + address"
    ),

    evaluate_blocking(
        name_fuzzy_address,
        "name_fuzzy + address"
    ),

    evaluate_blocking(
        name_union,
        "all_name"
    ),

    evaluate_blocking(
        all_name_address,
        "all_name + address"
    )
]


union_comparison = (
    pd.DataFrame(
        union_results
    )
)


union_comparison[
    "recall_percent"
] = (
    union_comparison[
        "blocking_recall"
    ] * 100
).round(3)


union_comparison

,method,true_pairs,retrieved_true_pairs,missed_true_pairs,blocking_recall,candidate_pairs,average_candidates_per_s1,median_candidates_per_s1,p95_candidates_per_s1,p99_candidates_per_s1,max_candidates_per_s1,recall_percent
0,name_char + address,3451,3413,38,0.988989,30421,30.421,26.0,78.0,80.0,80,98.899
1,name_fuzzy + address,3451,3405,46,0.986671,51135,51.135,50.0,79.0,80.0,80,98.667
2,all_name,3451,3291,160,0.953637,76944,76.944,84.0,102.0,114.0,130,95.364
3,all_name + address,3451,3438,13,0.996233,87933,87.933,90.0,128.0,137.0,147,99.623


In [86]:
# ============================================================
# COMPLETE BLOCKING COMPARISON
# ============================================================

blocking_comparison = pd.concat(
    [
        individual_comparison,
        union_comparison
    ],
    ignore_index=True
)


total_possible_pairs = (
    len(s1)
    *
    (
        len(s2)
        +
        len(s3)
    )
)


blocking_comparison[
    "candidate_reduction_percent"
] = (
    1
    -
    (
        blocking_comparison[
            "candidate_pairs"
        ]
        /
        total_possible_pairs
    )
) * 100


blocking_comparison = (
    blocking_comparison
    .sort_values(
        [
            "blocking_recall",
            "candidate_pairs"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(drop=True)
)


display(
    blocking_comparison[
        [
            "method",
            "true_pairs",
            "retrieved_true_pairs",
            "missed_true_pairs",
            "recall_percent",
            "candidate_pairs",
            "average_candidates_per_s1",
            "median_candidates_per_s1",
            "p95_candidates_per_s1",
            "p99_candidates_per_s1",
            "max_candidates_per_s1",
            "candidate_reduction_percent"
        ]
    ]
)

,method,true_pairs,retrieved_true_pairs,missed_true_pairs,recall_percent,candidate_pairs,average_candidates_per_s1,median_candidates_per_s1,p95_candidates_per_s1,p99_candidates_per_s1,max_candidates_per_s1,candidate_reduction_percent
0,all_name + address,3451,3438,13,99.623,87933,87.933,90.0,128.0,137.0,147,99.575326
1,name_char + address,3451,3413,38,98.899,30421,30.421,26.0,78.0,80.0,80,99.853081
2,name_fuzzy + address,3451,3405,46,98.667,51135,51.135,50.0,79.0,80.0,80,99.753043
3,all_name,3451,3291,160,95.364,76944,76.944,84.0,102.0,114.0,130,99.628398
4,name_fuzzy,3451,3097,354,89.742,39899,39.899,50.0,50.0,50.0,50,99.807307
5,address,3451,3060,391,88.670,14045,14.045,6.0,30.0,30.0,30,99.932169
6,name_char_tfidf,3451,3060,391,88.670,19128,19.128,11.0,50.0,50.0,50,99.907621
7,name_token,3451,2834,617,82.121,46730,46.730,50.0,50.0,50.0,50,99.774317


In [87]:
# ============================================================
# TRUE-PAIR RECOVERY MATRIX
# ============================================================

gt_recovery = gt_pairs.copy()


gt_recovery["pair_key"] = list(
    zip(
        gt_recovery["s1_entity_id"].astype(str),
        gt_recovery["true_entity_id"].astype(str)
    )
)


blocker_sets = {

    "address":
        make_pair_keys(
            address_candidates
        ),

    "name_token":
        make_pair_keys(
            name_token_candidates
        ),

    "name_char_tfidf":
        make_pair_keys(
            name_char_candidates
        ),

    "name_fuzzy":
        make_pair_keys(
            name_fuzzy_candidates
        )
}


for method, pair_set in blocker_sets.items():

    gt_recovery[
        f"retrieved_{method}"
    ] = gt_recovery[
        "pair_key"
    ].isin(pair_set)


gt_recovery["retrieved_by_name"] = (
    gt_recovery[
        [
            "retrieved_name_token",
            "retrieved_name_char_tfidf",
            "retrieved_name_fuzzy"
        ]
    ]
    .any(axis=1)
)


gt_recovery["retrieved_by_any"] = (
    gt_recovery[
        [
            "retrieved_address",
            "retrieved_name_token",
            "retrieved_name_char_tfidf",
            "retrieved_name_fuzzy"
        ]
    ]
    .any(axis=1)
)


print("=" * 80)
print("TRUE-PAIR RECOVERY")
print("=" * 80)


for method in blocker_sets:

    column = (
        f"retrieved_{method}"
    )

    count = int(
        gt_recovery[column].sum()
    )

    recall = (
        count / len(gt_recovery)
        if len(gt_recovery)
        else 0
    )

    print(
        f"{method:20s}: "
        f"{count:,} / "
        f"{len(gt_recovery):,} "
        f"({recall:.2%})"
    )


print()

print(
    "Recovered by any name blocker:",
    int(
        gt_recovery[
            "retrieved_by_name"
        ].sum()
    )
)


print(
    "Recovered by any blocker:",
    int(
        gt_recovery[
            "retrieved_by_any"
        ].sum()
    )
)


print(
    "Missed by all blockers:",
    int(
        (
            ~gt_recovery[
                "retrieved_by_any"
            ]
        ).sum()
    )
)

TRUE-PAIR RECOVERY
address             : 3,060 / 3,451 (88.67%)
name_token          : 2,834 / 3,451 (82.12%)
name_char_tfidf     : 3,060 / 3,451 (88.67%)
name_fuzzy          : 3,097 / 3,451 (89.74%)

Recovered by any name blocker: 3291
Recovered by any blocker: 3438
Missed by all blockers: 13


In [88]:
# ============================================================
# NAME VS ADDRESS CONTRIBUTION
# ============================================================

only_address = (
    gt_recovery[
        "retrieved_address"
    ]
    &
    ~gt_recovery[
        "retrieved_by_name"
    ]
)


only_name = (
    gt_recovery[
        "retrieved_by_name"
    ]
    &
    ~gt_recovery[
        "retrieved_address"
    ]
)


both_name_and_address = (
    gt_recovery[
        "retrieved_by_name"
    ]
    &
    gt_recovery[
        "retrieved_address"
    ]
)


missed_everything = (
    ~gt_recovery[
        "retrieved_by_any"
    ]
)


print("=" * 80)
print("BLOCKER CONTRIBUTION")
print("=" * 80)

print(
    f"Address only: "
    f"{only_address.sum():,}"
)

print(
    f"Name only: "
    f"{only_name.sum():,}"
)

print(
    f"Both name + address: "
    f"{both_name_and_address.sum():,}"
)

print(
    f"Missed by all: "
    f"{missed_everything.sum():,}"
)

BLOCKER CONTRIBUTION
Address only: 147
Name only: 378
Both name + address: 2,913
Missed by all: 13


In [89]:
# ============================================================
# MISSED TRUE PAIRS
# ============================================================

missed_pairs_v2 = (
    gt_recovery[
        ~gt_recovery[
            "retrieved_by_any"
        ]
    ]
    .copy()
)


missed_pairs_v2 = missed_pairs_v2[
    [
        "s1_entity_id",
        "true_entity_id",
        "retrieved_address",
        "retrieved_name_token",
        "retrieved_name_char_tfidf",
        "retrieved_name_fuzzy"
    ]
]


print("=" * 80)
print("MISSED TRUE PAIRS")
print("=" * 80)

print(
    f"Missed true pairs: "
    f"{len(missed_pairs_v2):,}"
)


missed_pairs_v2.to_csv(
    OUTPUT_DIR
    /
    "experiment2_missed_pairs.tsv",
    sep="\t",
    index=False
)


print(
    "Saved to:",
    OUTPUT_DIR
    /
    "experiment2_missed_pairs.tsv"
)

MISSED TRUE PAIRS
Missed true pairs: 13
Saved to: blocking_results\experiment2_missed_pairs.tsv


In [90]:
# ============================================================
# MISSED-PAIR INSPECTION
# ============================================================

def print_missed_pair(
    s1_id,
    true_id
):

    s1_row = s1_lookup[
        s1_id
    ]

    true_row = source_lookup[
        true_id
    ]

    name_similarity = (
        fuzz.ratio(
            s1_row["name_char_text"],
            true_row["name_char_text"]
        ) / 100
    )

    name_token_similarity = token_jaccard(
        s1_row["name_tokens"],
        true_row["name_tokens"]
    )

    core_name_similarity = token_jaccard(
        s1_row["name_core_tokens"],
        true_row["name_core_tokens"]
    )

    address_similarity = token_jaccard(
        s1_row["address_tokens"],
        true_row["address_tokens"]
    )

    print("=" * 80)

    print(
        f"S1: {s1_id}"
    )

    print(
        f"Name: {s1_row['business_name']}"
    )

    print(
        f"Address: {s1_row['business_address']}"
    )

    print(
        f"Country: {s1_row['country']}"
    )

    print()

    print(
        f"TRUE: {true_id}"
    )

    print(
        f"Name: {true_row['business_name']}"
    )

    print(
        f"Address: {true_row['business_address']}"
    )

    print(
        f"Country: {true_row['country']}"
    )

    print()

    print(
        f"Name fuzzy similarity: {name_similarity:.4f}"
    )

    print(
        f"Name token Jaccard: {name_token_similarity:.4f}"
    )

    print(
        f"Core name Jaccard: {core_name_similarity:.4f}"
    )

    print(
        f"Address token Jaccard: {address_similarity:.4f}"
    )


for _, pair in (
    missed_pairs_v2
    .head(30)
    .iterrows()
):

    print_missed_pair(
        pair["s1_entity_id"],
        pair["true_entity_id"]
    )

S1: S1-211535546
Name: International Infotech Pvt Ltd
Address: F-14 & F-15, First Floor F Block Inner Circle, Connaught Place, New Delhi, Central Delhi, Delhi
Country: India

TRUE: S3-419719124
Name: इंटरनेशनल इंफोटेक प्रा. लि.
Address: F-#14 & F-15, New Delhi, Central Delhi, DL
Country: India

Name fuzzy similarity: 0.5455
Name token Jaccard: 0.0000
Core name Jaccard: 0.0000
Address token Jaccard: 0.4286
S1: S1-347000383
Name: Sunrise Construction Private Limited
Address: 4Th Floor, C Wing, Trade World Kamala Mills Compound, S.B. Marg, Lower P, Arel, Mumbai, Mumbai City, Maharashtra
Country: India

TRUE: S2-389980705
Name: सनराइज कंस्ट्रक्शन प्राइवेट लिमिटेड
Address: H.NO D/4TH FLOOR, MUMBAI, MUMBAI CITY, महाराष्ट्र
Country: India

Name fuzzy similarity: 0.6866
Name token Jaccard: 0.0000
Core name Jaccard: 0.0000
Address token Jaccard: 0.1667
S1: S1-785596927
Name: Ganpati & Co
Address: Level 5, Dnyanvatsal Complex, Opp. Vandevi Mandir, Karve Nagar, Pune, Maharashtra
Country: India

T

In [91]:
# ============================================================
# FINAL CANDIDATE DISTRIBUTION
# ============================================================

final_candidate_pairs = (
    all_name_address
    .copy()
)


candidate_counts = (
    final_candidate_pairs
    .groupby(
        "s1_entity_id"
    )
    .size()
    .reindex(
        s1["entity_id"],
        fill_value=0
    )
)


print("=" * 80)
print("FINAL CANDIDATE DISTRIBUTION")
print("=" * 80)

print(
    f"Total candidate pairs: "
    f"{len(final_candidate_pairs):,}"
)

print(
    f"Average candidates/S1: "
    f"{candidate_counts.mean():.2f}"
)

print(
    f"Median candidates/S1: "
    f"{candidate_counts.median():.2f}"
)

print(
    f"P90 candidates/S1: "
    f"{candidate_counts.quantile(.90):.2f}"
)

print(
    f"P95 candidates/S1: "
    f"{candidate_counts.quantile(.95):.2f}"
)

print(
    f"P99 candidates/S1: "
    f"{candidate_counts.quantile(.99):.2f}"
)

print(
    f"Maximum candidates/S1: "
    f"{candidate_counts.max():,}"
)


print()

print(
    f"Cartesian pairs: "
    f"{total_possible_pairs:,}"
)


reduction_ratio = (
    1
    - (
        len(final_candidate_pairs)
        / total_possible_pairs
    )
)


print(
    f"Reduction ratio: {reduction_ratio:.4%}"
)

FINAL CANDIDATE DISTRIBUTION
Total candidate pairs: 87,933
Average candidates/S1: 87.93
Median candidates/S1: 90.00
P90 candidates/S1: 123.00
P95 candidates/S1: 128.00
P99 candidates/S1: 137.00
Maximum candidates/S1: 147

Cartesian pairs: 20,706,000
Reduction ratio: 99.5753%


In [92]:
# ============================================================
# CANDIDATE PROVENANCE
# ============================================================

all_block_rows = pd.concat(
    [
        address_candidates.assign(
            block_method="address"
        ),
        name_token_candidates.assign(
            block_method="name_token"
        ),
        name_char_candidates.assign(
            block_method="name_char_tfidf"
        ),
        name_fuzzy_candidates.assign(
            block_method="name_fuzzy"
        )
    ],
    ignore_index=True
)


candidate_provenance = (
    all_block_rows
    .groupby(
        [
            "s1_entity_id",
            "candidate_entity_id"
        ],
        as_index=False
    )
    .agg(
        blocking_methods=(
            "block_method",
            lambda values:
            ",".join(
                sorted(
                    set(values)
                )
            )
        ),

        num_blockers=(
            "block_method",
            "nunique"
        ),

        max_block_score=(
            "block_score",
            "max"
        )
    )
)


print(
    f"Candidate provenance rows: "
    f"{len(candidate_provenance):,}"
)


print(
    candidate_provenance[
        "blocking_methods"
    ]
    .value_counts()
    .head(20)
)

Candidate provenance rows: 87,933
blocking_methods
name_token                                       30124
name_fuzzy                                       23057
address                                          10989
name_char_tfidf,name_fuzzy,name_token             6444
name_fuzzy,name_token                             4331
name_char_tfidf                                   3447
name_char_tfidf,name_fuzzy                        3258
name_char_tfidf,name_token                        3227
address,name_char_tfidf,name_fuzzy,name_token     2366
address,name_char_tfidf,name_fuzzy                 229
address,name_fuzzy                                 174
address,name_char_tfidf,name_token                 108
address,name_token                                  90
address,name_char_tfidf                             49
address,name_fuzzy,name_token                       40
Name: count, dtype: int64


In [93]:
# ============================================================
# SAVE EXPERIMENT 2 OUTPUTS
# ============================================================

FINAL_CANDIDATE_FILE = (
    OUTPUT_DIR
    /
    "experiment2_candidate_pairs.tsv"
)


PROVENANCE_FILE = (
    OUTPUT_DIR
    /
    "experiment2_candidate_provenance.tsv"
)


COMPARISON_FILE = (
    OUTPUT_DIR
    /
    "experiment2_blocking_comparison.tsv"
)


MISSED_FILE = (
    OUTPUT_DIR
    /
    "experiment2_missed_pairs.tsv"
)


final_candidate_pairs[
    [
        "s1_entity_id",
        "candidate_entity_id"
    ]
].drop_duplicates().to_csv(
    FINAL_CANDIDATE_FILE,
    sep="\t",
    index=False
)


candidate_provenance.to_csv(
    PROVENANCE_FILE,
    sep="\t",
    index=False
)


blocking_comparison.to_csv(
    COMPARISON_FILE,
    sep="\t",
    index=False
)


missed_pairs_v2.to_csv(
    MISSED_FILE,
    sep="\t",
    index=False
)


print("=" * 80)
print("FILES SAVED")
print("=" * 80)

print(
    FINAL_CANDIDATE_FILE
)

print(
    PROVENANCE_FILE
)

print(
    COMPARISON_FILE
)

print(
    MISSED_FILE
)

FILES SAVED
blocking_results\experiment2_candidate_pairs.tsv
blocking_results\experiment2_candidate_provenance.tsv
blocking_results\experiment2_blocking_comparison.tsv
blocking_results\experiment2_missed_pairs.tsv


In [94]:
# ============================================================
# FINAL SANITY CHECKS
# ============================================================

print("=" * 80)
print("FINAL SANITY CHECKS")
print("=" * 80)


# ------------------------------------------------------------
# 1. Candidate IDs must be S2/S3
# ------------------------------------------------------------

candidate_ids = set(
    final_candidate_pairs[
        "candidate_entity_id"
    ]
)


valid_candidate_ids = set(
    s2["entity_id"]
).union(
    set(
        s3["entity_id"]
    )
)


invalid_candidate_ids = (
    candidate_ids
    -
    valid_candidate_ids
)


print(
    f"Invalid candidate IDs: "
    f"{len(invalid_candidate_ids):,}"
)


if invalid_candidate_ids:

    raise ValueError(
        "Candidate set contains IDs "
        "that are not from S2/S3."
    )


# ------------------------------------------------------------
# 2. Candidate pair uniqueness
# ------------------------------------------------------------

duplicate_pairs = (
    final_candidate_pairs
    .duplicated(
        [
            "s1_entity_id",
            "candidate_entity_id"
        ]
    )
    .sum()
)


print(
    f"Duplicate candidate pairs: "
    f"{duplicate_pairs:,}"
)


if duplicate_pairs:

    raise ValueError(
        "Duplicate candidate pairs found."
    )


# ------------------------------------------------------------
# 3. S1 IDs
# ------------------------------------------------------------

invalid_s1_ids = (
    set(
        final_candidate_pairs[
            "s1_entity_id"
        ]
    )
    -
    set(
        s1["entity_id"]
    )
)


print(
    f"Invalid S1 IDs: "
    f"{len(invalid_s1_ids):,}"
)


if invalid_s1_ids:

    raise ValueError(
        "Candidate set contains "
        "unknown S1 IDs."
    )


# ------------------------------------------------------------
# 4. Blocking recall
# ------------------------------------------------------------

final_eval = evaluate_blocking(
    final_candidate_pairs,
    "final_experiment2"
)


print(
    f"Final blocking recall: "
    f"{final_eval['blocking_recall']:.4%}"
)


print(
    f"Final candidate pairs: "
    f"{final_eval['candidate_pairs']:,}"
)


print(
    f"Average candidates/S1: "
    f"{final_eval['average_candidates_per_s1']:.2f}"
)


print(
    f"P95 candidates/S1: "
    f"{final_eval['p95_candidates_per_s1']:.2f}"
)


print(
    f"P99 candidates/S1: "
    f"{final_eval['p99_candidates_per_s1']:.2f}"
)


print("\nAll sanity checks passed.")

FINAL SANITY CHECKS
Invalid candidate IDs: 0
Duplicate candidate pairs: 0
Invalid S1 IDs: 0
Final blocking recall: 99.6233%
Final candidate pairs: 87,933
Average candidates/S1: 87.93
P95 candidates/S1: 128.00
P99 candidates/S1: 137.00

All sanity checks passed.
